# Análise de Projetos de Investimento - Distrito Federal

## 1. Introdução e Contexto
## 2. Extração de Dados
## 3. Análise Exploratória Inicial
## 4. Tratamento e Transformação
## 5. Análise Quantitativa
## 6. Análise Qualitativa
## 7. Visualizações
## 8. Conclusões e Recomendações

In [5]:
# Importar bibliotecas necessárias
import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
import time
from datetime import datetime
import json
from tqdm import tqdm


# Documentação de Configurações

## Fonte dos Dados

Os dados utilizados nesta análise foram extraídos da **API ObrasGov.br**, especificamente do endpoint `/projeto-investimento`.

- **URL Base da API:** `https://api.obrasgov.gestao.gov.br/obrasgov/api/projeto-investimento`
- **Filtro Aplicado:** A extração foi restrita aos projetos localizados na **Unidade Federativa (UF) do Distrito Federal (DF)**, utilizando o parâmetro `uf=DF`.
- **Formato:** Os dados são retornados em formato JSON, com estrutura aninhada (listas de dicionários) para campos como executores, fontes de recurso e tipos de projeto.

## Objetivo da Análise

O objetivo principal desta análise é demonstrar a capacidade de construir um pipeline de processamento de dados (ETL - Extração, Transformação e Carga) e realizar uma análise exploratória sobre dados públicos de projetos de investimento.

**Objetivos Específicos:**

1. Integrar-se com uma API pública (ObrasGov.br) para extrair dados de forma paginada.
2. Tratar e normalizar dados complexos.
3. Persistir os dados tratados em um banco de dados relacional (SQLite).
4. Gerar insights e visualizações que respondam a perguntas de negócio sobre os investimentos no DF.




In [6]:
# 1. Definir constantes (URL base, parâmetros)
BASE_URL = "https://api.obrasgov.gestao.gov.br/obrasgov/api/projeto-investimento"
DEFAULT_PARAMS = {
    "uf": "DF",
    "size": 1000, # Tamanho máximo por página
    "page": 0
}
MAX_RETRIES = 3
RETRY_DELAY = 5 # segundos

# 2. Função para fazer requisição com retry
def fetch_page_with_retry(page_num, max_retries=MAX_RETRIES, retry_delay=RETRY_DELAY):
    '''Faz a requisição para uma página específica da API com lógica de retry.'''
    params = DEFAULT_PARAMS.copy()
    params['page'] = page_num
    
    for attempt in range(max_retries):
        try:
            response = requests.get(BASE_URL, params=params, timeout=30)
            response.raise_for_status() # Levanta exceção para códigos de erro HTTP
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Erro na requisição da página {page_num} (Tentativa {attempt + 1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            else:
                print(f"Falha final ao buscar a página {page_num} após {max_retries} tentativas.")
                return None

# 3. Função para paginação
def fetch_all_data():
    '''Implementa a lógica de paginação para extrair todos os dados.'''
    all_data = []
    page_num = 0
    total_pages = 1 # Inicializa com 1 para entrar no loop

    print("\nIniciando extração de dados da API ObrasGov.br (UF=DF)...")

    # Tenta buscar a primeira página para obter o total de páginas
    initial_data = fetch_page_with_retry(0)
    if initial_data and 'content' in initial_data:
        all_data.extend(initial_data['content'])
        total_pages = initial_data.get('totalPages', 1)
        page_num = 1
    elif initial_data is None:
        print("Não foi possível obter a primeira página. Abortando extração.")
        return []
    else:
        print("A primeira página não contém dados ou tem formato inesperado. Abortando extração.")
        return []

    # Continua a busca para as páginas restantes
    with tqdm(total=total_pages, initial=1, desc="Páginas", unit="pág") as pbar:
        while page_num < total_pages:
            data = fetch_page_with_retry(page_num)
            
            if data and 'content' in data:
                all_data.extend(data['content'])
                page_num += 1
                pbar.update(1)
            else:
                print(f"Parando a extração na página {page_num} devido a erro ou falta de dados.")
                break
                
    print(f"Extração concluída. Total de {len(all_data)} registros obtidos.")
    return all_data

In [7]:
# Cria o diretório para salvar os dados brutos, se não existir
os.makedirs("data/raw", exist_ok=True)

# 4. Executar extração completa
dados_brutos = fetch_all_data()

# Salvar dados brutos
if dados_brutos:
    raw_file_path = 'data/raw/projetos_df_raw.json'
    with open(raw_file_path, 'w', encoding='utf-8') as f:
        json.dump(dados_brutos, f, ensure_ascii=False, indent=4)
    print(f"\nDados brutos salvos em: {raw_file_path}")
    print(f"Total de registros: {len(dados_brutos)}")
    
    # Mostrar amostra dos dados
    print("\nAmostra do primeiro registro:")
    # Usamos [0] para pegar o primeiro elemento da lista de projetos
    print(json.dumps(dados_brutos[0], indent=2)) 
else:
    print("Nenhum dado para salvar.")


Iniciando extração de dados da API ObrasGov.br (UF=DF)...


Páginas: 100%|██████████| 1/1 [00:00<?, ?pág/s]

Extração concluída. Total de 10 registros obtidos.

Dados brutos salvos em: data/raw/projetos_df_raw.json
Total de registros: 10

Amostra do primeiro registro:
{
  "idUnico": "50379.53-54",
  "nome": "DL - 304/2024 - Contrata\u00e7\u00e3o de institui\u00e7\u00e3o para execu\u00e7\u00e3o de servi\u00e7os t\u00e9cnico-especializados para realiza\u00e7\u00e3o de atualiza\u00e7\u00f5es no M\u00e9todo de Dimensionamento de Pavimentos R\u00edgidos do DNI",
  "cep": null,
  "endereco": null,
  "descricao": "Contrata\u00e7\u00e3o de institui\u00e7\u00e3o para execu\u00e7\u00e3o de servi\u00e7os t\u00e9cnico-especializados para realiza\u00e7\u00e3o de atualiza\u00e7\u00f5es no M\u00e9todo de Dimensionamento de Pavimentos R\u00edgidos do DNI",
  "funcaoSocial": "Amplia\u00e7\u00e3o da capacidade de trafego visando a melhoria da seguran\u00e7a do usu\u00e1rio",
  "metaGlobal": "Projetos B\u00e1sicos e Executivos de Engenharia",
  "dataInicialPrevista": "2024-12-20",
  "dataFinalPrevista": "2027-1